# Libraries

In [23]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dense,
    Attention,
    Add,
    LayerNormalization,
    PReLU
)

from tensorflow.keras.regularizers import l2
from tensorflow.keras.initializers import (
    LecunUniform,
    Orthogonal,
    Zeros
)

from sklearn.utils.class_weight import compute_class_weight

# Load dataset

In [2]:
data = np.load('./data/financial_dataset.npz')

X = data['X']
y_regression = data['y_regression']
y_classification = data['y_classification']
sequence_dates = data['sequence_dates']

# Create model

In [3]:
def build_attention_lstm_model(
    seq_len=20,
    n_features=5
):

    inputs = Input(
        shape=(seq_len, n_features)
    )

    # =====================================================
    # SHARED BACKBONE
    # =====================================================

    x = LSTM(
        units=64,
        return_sequences=True,
        activation='tanh',
        recurrent_activation='sigmoid',
        dropout=0.2,
        recurrent_dropout=0.2,
        kernel_regularizer=l2(1e-6),
        recurrent_regularizer=l2(1e-6),
        kernel_initializer=LecunUniform(),
        recurrent_initializer=Orthogonal(),
        bias_initializer=Zeros()
    )(inputs)

    attention_output = Attention()([x, x])

    x = Add()([x, attention_output])

    x = LayerNormalization()(x)

    x = LSTM(
        units=32,
        return_sequences=False,
        activation='tanh',
        recurrent_activation='sigmoid',
        dropout=0.2,
        kernel_regularizer=l2(1e-6),
        recurrent_regularizer=l2(1e-6),
        kernel_initializer=LecunUniform(),
        recurrent_initializer=Orthogonal(),
        bias_initializer=Zeros()
    )(x)

    x = Dense(16)(x)
    x = PReLU()(x)

    shared_representation = Dense(8)(x)
    shared_representation = PReLU()(shared_representation)

    # =====================================================
    # REGRESSION HEAD
    # =====================================================

    regression_output = Dense(
        1,
        activation='linear',
        name='regression_output'
    )(shared_representation)

    # =====================================================
    # CLASSIFICATION HEAD
    # =====================================================

    classification_output = Dense(
        1,
        activation='sigmoid',
        name='classification_output'
    )(shared_representation)

    # =====================================================
    # MODEL
    # =====================================================

    model = Model(
        inputs=inputs,
        outputs=[
            regression_output,
            classification_output
        ]
    )

    return model

In [4]:
model = build_attention_lstm_model()

2026-05-21 14:56:17.539000: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-05-21 14:56:17.539257: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-05-21 14:56:17.539273: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-05-21 14:56:17.539580: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-05-21 14:56:17.539611: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [6]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 20, 5)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 20, 64)    │     17,920 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 20, 64)    │          0 │ lstm[0][0],       │
│ (Attention)         │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 20, 64)    │          0 │ lstm[0][0],       │
│                     │                   │            │ attention[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 20, 64)    │        128 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 32)        │     12,416 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 16)        │        528 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ p_re_lu (PReLU)     │ (None, 16)        │         16 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 8)         │        136 │ p_re_lu[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ p_re_lu_1 (PReLU)   │ (None, 8)         │          8 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ regression_output   │ (None, 1)         │          9 │ p_re_lu_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ classification_out… │ (None, 1)         │          9 │ p_re_lu_1[0][0]   │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 31,170 (121.76 KB)

 Trainable params: 31,170 (121.76 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
model.compile(

    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),

    loss=[

        'mse',

        tf.keras.losses.BinaryFocalCrossentropy()

    ],

    loss_weights=[

        1.0,

        0.5

    ],

    metrics=[

        ['mae'],

        [
            'accuracy',
            tf.keras.metrics.Precision(),
            tf.keras.metrics.Recall()
        ]

    ]
)

# Train model

## Split data

In [11]:
train_mask = sequence_dates < np.datetime64('2017-01-01')

val_mask = (
    (sequence_dates >= np.datetime64('2017-01-01')) &
    (sequence_dates < np.datetime64('2018-01-01'))
)

test_mask = sequence_dates >= np.datetime64('2018-01-01')

# TRAIN
X_train = X[train_mask]
y_reg_train = y_regression[train_mask]
y_cls_train = y_classification[train_mask]


# VALIDATION

X_val = X[val_mask]
y_reg_val = y_regression[val_mask]
y_cls_val = y_classification[val_mask]

# TEST

X_test = X[test_mask]
y_reg_test = y_regression[test_mask]
y_cls_test = y_classification[test_mask]

# INFO

print("\n====================")
print("TRAIN")
print("====================")

print(X_train.shape)
print(y_reg_train.shape)
print(y_cls_train.shape)

print("\n====================")
print("VALIDATION")
print("====================")

print(X_val.shape)
print(y_reg_val.shape)
print(y_cls_val.shape)

print("\n====================")
print("TEST")
print("====================")

print(X_test.shape)
print(y_reg_test.shape)
print(y_cls_test.shape)


TRAIN
(460790, 20, 5)
(460790,)
(460790,)

VALIDATION
(123218, 20, 5)
(123218,)
(123218,)

TEST
(12766, 20, 5)
(12766,)
(12766,)


In [22]:


classes = np.unique(y_cls_train)

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_cls_train
)

class_weights = dict(zip(classes, weights))

print(class_weights)


regression_weights = np.ones(len(y_reg_train))

classification_weights = np.array([
    class_weights[int(label)]
    for label in y_cls_train.ravel()
])

history = model.fit(

    X_train,

    [
        y_reg_train,
        y_cls_train
    ],

    sample_weight=[

        regression_weights,
        classification_weights
    ],

    validation_data=(

        X_val,

        [
            y_reg_val,
            y_cls_val
        ]

    ),

    epochs=100,
    batch_size=256,
    shuffle=False,
    verbose=1
)

{0: 1.2943466609738148, 1: 0.814724052208537}
Epoch 1/100
 319/1800 ━━━━━━━━━━━━━━━━━━━━ 11:13 455ms/step - classification_output_accuracy: 0.5928 - classification_output_loss: 0.1819 - classification_output_precision_3: 0.6477 - classification_output_recall_3: 0.8046 - loss: 18.6239 - regression_output_loss: 18.5322 - regression_output_mae: 2.7177

KeyboardInterrupt: 